In [2]:
!pip install -qU bertopic pandas nltk scikit-learn matplotlib xlwt 


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


## 1. Imports

In [1]:
# ============================================
# 0. CONFIGURATION & PARAMETERS
# ============================================

# --- Data paths ---
DATA_PATH = '/Users/Axel/Documents/github/wsm/data/ryr.csv'

# --- Column names in the CSV ---
TEXT_COLUMN = 'Abstract'
DATE_COLUMN = 'Date'

# --- Embedding model ---
SENTENCE_MODEL_NAME = 'all-MiniLM-L12-v2'

# --- UMAP parameters ---
UMAP_N_NEIGHBORS = 20
UMAP_N_COMPONENTS = 5
UMAP_MIN_DIST = 0.0
UMAP_METRIC = 'cosine'

# --- HDBSCAN parameters ---
HDBSCAN_MIN_CLUSTER_SIZE = 10
HDBSCAN_MIN_SAMPLES = 1
HDBSCAN_METRIC = 'euclidean'
HDBSCAN_CLUSTER_SELECTION_METHOD = 'leaf'
HDBSCAN_CLUSTER_SELECTION_EPSILON = 0.1

# --- Model saving ---
TOPIC_MODEL_SAVE_PATH = '/Users/Axel/Documents/github/wsm/data/bertopic_model_ryr'

In [2]:
# ============================================
# 1. IMPORTS
# ============================================

import re
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from umap import UMAP
import hdbscan
from bertopic import BERTopic

import plotly.express as px


/Users/Axel/Documents/github/wsm/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load & Preprocess Data

In [3]:
# ============================================
# 2. LOAD & PREPROCESS DATA
# ============================================

# Load CSV
docs = pd.read_csv(DATA_PATH)

# Basic cleaning: require both text and date
docs = docs.dropna(subset=[TEXT_COLUMN, DATE_COLUMN])
docs = docs.reset_index(drop=True)
print(f"Loaded {len(docs)} rows after initial dropna().")

# Ensure text column is string
docs[TEXT_COLUMN] = docs[TEXT_COLUMN].astype(str)

# ---- Count before date parsing ----
count_before_date_parse = len(docs)

# Parse dates
docs[DATE_COLUMN] = pd.to_datetime(docs[DATE_COLUMN], errors='coerce')

# Drop rows with unparseable dates
docs = docs.dropna(subset=[DATE_COLUMN]).reset_index(drop=True)

# ---- Count after date parsing ----
count_after_date_parse = len(docs)

print(f"Rows before parsing dates:  {count_before_date_parse}")
print(f"Rows after date parsing:   {count_after_date_parse}")
print(f"Dropped due to bad dates:  {count_before_date_parse - count_after_date_parse}")

# Extract Year
docs["Year"] = docs[DATE_COLUMN].dt.year

# Publications per year
pubs_per_year = docs["Year"].value_counts().sort_index()
print("Publications per year:")
print(pubs_per_year)


Loaded 7144 rows after initial dropna().
Rows before parsing dates:  7144
Rows after date parsing:   7144
Dropped due to bad dates:  0
Publications per year:
Year
2009      64
2010     535
2011     631
2012     736
2013     879
2014     957
2015    1047
2016    1159
2017    1124
2018      12
Name: count, dtype: int64


In [4]:
# Count before filtering
count_before_year_filter = len(docs)

# Keep only publications between 2004 and 2007 (inclusive)
docs = docs[(docs["Year"] >= 2010) & (docs["Year"] <= 2017)].reset_index(drop=True)

# Count after filtering
count_after_year_filter = len(docs)

print(f"Rows before year filtering: {count_before_year_filter}")
print(f"Rows after year filtering:  {count_after_year_filter}")
print(f"Rows dropped:               {count_before_year_filter - count_after_year_filter}")

Rows before year filtering: 7144
Rows after year filtering:  7068
Rows dropped:               76


In [5]:
# Prepare the abstracts as a list for BERTopic
text = docs[TEXT_COLUMN].astype(str).tolist()
len(text)

7068

## 3. Embeddings & Topic Modeling (UMAP + HDBSCAN + BERTopic)

In [ ]:
# ============================================
# 3. LOAD PRE-TRAINED TOPIC MODEL
# ============================================

from bertopic import BERTopic

print("Loading pre-trained topic model...")
topic_model = BERTopic.load(TOPIC_MODEL_SAVE_PATH)

topic_info = topic_model.get_topic_info()
print(f"Number of topics (including outlier -1): {len(topic_info)}")
topic_info.head()


In [21]:
pattern = r"\bryr\b"
mask = topic_info["Representation"].astype(str).str.contains(
    pattern, flags=re.IGNORECASE, regex=True
)

pfas_topics = topic_info[mask]
print(f"Number of topics containing 'ryr': {len(pfas_topics)}")

Number of topics containing 'ryr': 1


In [22]:
import pprint
pp = pprint.PrettyPrinter(width=120, compact=False)

for idx, row in pfas_topics.iterrows():
    print(f"\n=== Topic {row['Topic']} ===")
    print("Name:", row.get("Name", ""))
    print("Count:", row.get("Count", ""))
    print("Representation:")
    pp.pprint(row["Representation"])



=== Topic 79 ===
Name: 79_rice_red_yeast_lovastatin
Count: 26
Representation:
['rice', 'red', 'yeast', 'lovastatin', 'citrinin', 'ryr', 'monacolin', 'monacolins', 'fermented', 'bsm']


## 4. Save Topic Model (optional)

In [ ]:
# ============================================
# 4. LOAD TOPIC MODEL (COMPLETED)
# ============================================

print(f"Model successfully loaded from: {TOPIC_MODEL_SAVE_PATH}")


In [37]:
# ============================================
# 5. BUILD DOC_INFO WITH DATES
# ============================================

# Make a clean copy of docs and ensure alignment
docs_model = docs.reset_index(drop=True).copy()
assert len(text) == len(docs_model), "len(text) must match len(docs)."

# Get document-level topic assignments from BERTopic
doc_info = topic_model.get_document_info(text).copy()

# Create a stable doc_id BEFORE any filtering
doc_info["doc_id"] = np.arange(len(doc_info))
docs_model["doc_id"] = np.arange(len(docs_model))

# Merge Date from docs_model into doc_info
doc_info = doc_info.merge(
    docs_model[["doc_id", DATE_COLUMN]],
    on="doc_id",
    how="left"
).rename(columns={DATE_COLUMN: "Date"})

# Drop outlier topic -1 AFTER alignment
doc_info = doc_info[doc_info["Topic"] != -1].copy()

# Ensure datetime
doc_info["Date"] = pd.to_datetime(doc_info["Date"], errors="coerce")

# Drop rows with missing dates (optional)
doc_info = doc_info.dropna(subset=["Date"]).copy()

# Convenience ID column
doc_info["ID"] = doc_info["doc_id"]

print(f"Documents available (no cutoff applied): {len(doc_info)}")
doc_info.head()


Documents available (no cutoff applied): 6149


Document  \
0                                                                                                                                                                                                                                                                                                                       Autism is a neurodevelopmental disorder characterized by impairments in communication and reciprocal social interaction, coupled with repetitive behavior, which typically manifests by 3 years of age. Multiple genes and early exposure to environmental factors are the etiological determinants of the disorder that contribute to variable expression of autism-related traits. Increasing evidence indicates that altered fatty acid metabolic pathways may affect proper function of the nervous system and contribute to autism spectrum disorders. This review provides an overview of the reported abnormalities associated with the synthesis of membrane fatty acids in individuals with autism as a result of insufficient dietary supplementation or genetic defects. Moreover, we discuss deficits associated with the release of arachidonic acid from the membrane phospholipids and its subsequent metabolism to bioactive prostaglandins via phospholipase A(2)-cyclooxygenase biosynthetic pathway in autism spectrum disorders. The existing evidence for the involvement of lipid neurobiology in the pathology of neurodevelopmental disorders such as autism is compelling and opens up an interesting possibility for further investigation of this metabolic pathway.   
1                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            Pancreatic cancer is a malignancy of poor prognosis which is mostly diagnosed at advanced stages. Current treatment modalities are very limited creating great interest for novel preventive and therapeutic options. Vitamin D seems to have a protective effect against pancreatic cancer by participating in numerous proapoptotic, antiangiogenic, anti-inflammatory, prodifferentiating, and immunomodulating mechanisms. 25-hydroxyvitamin D [25(OH)D] serum concentrations are currently the best indicator of vitamin D status. There are three main sources of vitamin D: sun exposure, diet,and dietary supplements. Sun exposure has been associated with lower incidence of pancreatic cancer in ecological studies. Increased vitamin D levels seem to protect against pancreatic cancer, but caution is needed as excessive dietary intake may have opposite results. Future studies will verify the role of vitamin D in the prevention and therapy of pancreatic cancer and will lead to guidelines on adequate sun exposure and vitamin D dietary intake.   
3                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [38]:
def list_contains_ryr(rep_list):
    if not isinstance(rep_list, list):
        return False
    return any("ryr" in str(item).lower() for item in rep_list)

mask_ryr = doc_info["Representation"].apply(list_contains_ryr)
df_ryr = doc_info[mask_ryr].copy()

print("ryr publications:", len(df_ryr))

ryr publications: 26


In [39]:
df_ryr["Year"] = df_ryr["Date"].dt.year
ryr_per_year = df_ryr["Year"].value_counts().sort_index()
print(ryr_per_year)

Year
2012     2
2013     5
2014     3
2016     5
2017    11
Name: count, dtype: int64


In [40]:
df_ryr["YearMonth"] = df_ryr["Date"].dt.to_period("M")

In [41]:
ryr_per_month = df_ryr["YearMonth"].value_counts().sort_index()
all_months = pd.period_range(
    df_ryr["Date"].min().to_period("M"),
    df_ryr["Date"].max().to_period("M"),
    freq="M"
)

ryr_per_month_full = ryr_per_month.reindex(all_months, fill_value=0)
display(ryr_per_month_full)


2012-11    2
2012-12    0
2013-01    2
2013-02    0
2013-03    0
          ..
2017-08    2
2017-09    0
2017-10    0
2017-11    0
2017-12    2
Freq: M, Name: count, Length: 62, dtype: int64

## 6. Topic Metadata (`Name` & `Representation`)

In [42]:
# ============================================
# 6. TOPIC METADATA (Name & Representation)
# ============================================

# Ensure we have a Name column for each document's topic
if 'Name' not in doc_info.columns or doc_info['Name'].isna().all():
    labels = topic_model.generate_topic_labels()
    doc_info['Name'] = doc_info['Topic'].map(labels)

# Get topic_info and ensure Representation is present
try:
    topic_info = topic_model.get_topic_info()
    if 'Representation' not in topic_info.columns:
        topic_info['Representation'] = topic_info['Name']
except Exception:
    topic_info = pd.DataFrame({'Name': doc_info['Name'].unique()})
    topic_info['Count'] = (
        doc_info['Name']
        .value_counts()
        .reindex(topic_info['Name'])
        .fillna(0)
        .astype(int)
    )
    topic_info['Representation'] = topic_info['Name']

topic_info.head()


Topic  Count                                      Name  \
0     -1    919                          -1_and_the_of_in   
1      0    145        0_ad_cognitive_alzheimers_dementia   
2      1    126              1_amd_macular_eye_agerelated   
3      2    112                         2_n3_pufa_epa_oil   
4      3    109  3_ons_malnutrition_nutritional_residents   

                                                                                    Representation  \
0                                                [and, the, of, in, to, as, with, for, were, that]   
1      [ad, cognitive, alzheimers, dementia, memory, disease, amyloid, decline, impairment, brain]   
2       [amd, macular, eye, agerelated, retinal, degeneration, areds, lutein, progression, visual]   
3                                      [n3, pufa, epa, oil, fatty, dha, fish, omega3, acids, acid]   
4  [ons, malnutrition, nutritional, residents, care, nutrition, home, hospital, patients, nursing]   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

## 7. Topic Share by Time Bucket (M/W/D)

In [ ]:
# ============================================
# 7. TOPIC SHARE BY HALF-YEAR BUCKET (H1/H2)
# ============================================

import numpy as np
import pandas as pd

def shares_by_halfyear(df: pd.DataFrame) -> pd.DataFrame:
    tmp = df.copy()

    # Define clean, non-overlapping half-year labels like "2010H1", "2010H2"
    tmp["HalfYear"] = (
        tmp["Date"].dt.year.astype(str)
        + "H"
        + np.where(tmp["Date"].dt.month <= 6, "1", "2")
    )

    # Count documents per (HalfYear, Topic Name)
    counts = pd.pivot_table(
        tmp,
        index="HalfYear",
        columns="Name",
        aggfunc="size",
        fill_value=0,
    )

    # Convert to per-half-year topic shares
    row_sums = counts.sum(axis=1).replace(0, 1)
    shares = counts.div(row_sums, axis=0)
    return shares

# Compute half-year topic shares for the entire corpus
shares = shares_by_halfyear(doc_info)
chosen = "H2"   # custom label indicating half-year buckets

print("Chosen time bucket: HalfYear (H1/H2)")
display(shares.head())


## 8. Compute Recency Score & Recency + Magnitude Score

In [ ]:
# ============================================
# 8. RECENCY+JUMP SCORES
# ============================================

# --- Time decay for recency scoring ---
HALF_LIFE = 1.0                # in units of the chosen bucket (M/W/D)

# --- Magnitude (jump) weighting ---
MAG_ALPHA = 4.0                # strength of magnitude contribution
RECENT_WINDOW = 2              # number of most recent buckets to treat as "recent history"

import numpy as np
import pandas as pd

if shares is None or chosen is None:
    raise RuntimeError("Time-bucketed topic shares not available. Run the previous cell that computes `shares` and `chosen`.")

# Ensure temporal ordering
shares_sorted = shares.sort_index()
n_buckets = len(shares_sorted)

# 1) RECENCY SCORE (EXPONENTIAL TIME DECAY)
ages = np.arange(n_buckets - 1, -1, -1, dtype=float)
w = 0.5 ** (ages / max(HALF_LIFE, 1e-6))
w = w / w.sum()
recency_score = (shares_sorted.mul(w, axis=0)).sum(axis=0)

# 2) MAGNITUDE TERM: RECENT vs HISTORICAL MEAN
if n_buckets > RECENT_WINDOW:
    recent_mean = shares_sorted.tail(RECENT_WINDOW).mean(axis=0)
    past_mean = shares_sorted.iloc[:-RECENT_WINDOW].mean(axis=0)
else:
    recent_mean = shares_sorted.tail(1).iloc[0]
    if n_buckets > 1:
        past_mean = shares_sorted.iloc[:-1].mean(axis=0)
    else:
        past_mean = pd.Series(0.0, index=shares_sorted.columns)

jump_raw = recent_mean - past_mean
jump_strength = jump_raw  # can be further transformed if needed

# 3) LIFETIME PENALTY
lifetime = (shares_sorted > 0).sum(axis=0).astype(float)
penalty = 1.0 / np.log1p(lifetime)
penalty.replace([np.inf, -np.inf], 0.0, inplace=True)
penalty = penalty.fillna(0.0)

# 4) FINAL RECENCY + MAGNITUDE SCORE
recency_jump_score = (recency_score + MAG_ALPHA * jump_strength) * penalty

# Align indices (safety)
recency_score = recency_score.sort_index()
jump_strength = jump_strength.reindex(recency_score.index).fillna(0.0)
recency_jump_score = recency_jump_score.reindex(recency_score.index).fillna(0.0)

print(
    f"Computed recency_score (half-life={HALF_LIFE}, freq='{chosen}'), "
    f"magnitude term (recent vs historical mean, window={RECENT_WINDOW}), "
    "and combined recency_jump_score with lifetime penalty."
)

display(
    pd.DataFrame({
        "recency_score": recency_score,
        "jump_strength": jump_strength,
        "recency_jump_score": recency_jump_score,
    }).head()
)


## 9. Build Plotting DataFrame

In [ ]:
# ============================================
# 9. BUILD PLOTTING DATAFRAMES (RECENCY VS RECENCY+JUMP)
# ============================================

# Base topic meta
topic_meta = topic_info[["Name", "Count", "Representation"]].copy()

# --- Recency + jump df_plot_jump ---
df_plot_jump = (
    recency_jump_score
    .rename("recency_jump_score")
    .reset_index()
    .rename(columns={"index": "Name"})
)

df_extra = pd.DataFrame({
    "Name": recency_score.index,
    "recency_score": recency_score.values,
    "jump_strength": jump_strength.values,
})

df_plot_jump = df_plot_jump.merge(df_extra, on="Name", how="left")
df_plot_jump = df_plot_jump.merge(topic_meta, on="Name", how="left")

if ("Count" not in df_plot_jump.columns) or df_plot_jump["Count"].isna().all():
    counts = (
        doc_info["Name"]
        .value_counts()
        .reindex(df_plot_jump["Name"])
        .fillna(0)
        .astype(int)
    )
    df_plot_jump["Count"] = counts.values

display(df_plot_jump.head())


## 10. Visualize Topics & Highlight ryr

In [46]:
import plotly.graph_objects as go
import re

# --- Boolean mask: Representation contains the token "ryr" ---
mask_ryr = df_plot_jump["Representation"].apply(
    lambda lst: any(re.fullmatch(r"ryr", str(x).lower()) for x in lst)
)

df_main = df_plot_jump[~mask_ryr]
df_highlight = df_plot_jump[mask_ryr]

fig = go.Figure()

# --- Base layer: all topics ---
fig.add_trace(
    go.Scatter(
        x=df_main["Count"],
        y=df_main["recency_jump_score"],
        mode="markers",
        marker=dict(size=8, opacity=0.7),
        text=df_main["Name"],
        hovertext=df_main["Representation"].astype(str),
        name="All topics",
    )
)

# --- Highlighted: topics containing the exact token "ryr" ---
fig.add_trace(
    go.Scatter(
        x=df_highlight["Count"],
        y=df_highlight["recency_jump_score"],
        mode="markers",
        marker=dict(
            size=14,
            color="red",
            line=dict(width=2, color="black")
        ),
        text=df_highlight["Name"],
        hovertext=df_highlight["Representation"].astype(str),
        name="RYR-related topic",
    )
)

# Axis + layout
fig.update_xaxes(title_text="Count")
fig.update_yaxes(title_text="recency_jump_score")

fig.update_layout(
    title="Weak-signal score: recency + magnitude (highlighting ryr topic)"
)

fig.show()


## 11. Top-k Weak Signals

In [ ]:
# --- Weak-signal ranking parameters ---
TOP_K_WEAK = 10                  # maximum number of weak signals to present
WEAK_MIN_COUNT = 10              # minimum topic Count to consider
WEAK_MIN_UPPER_COUNT = 30        # absolute upper bound for Count
SCORE_MIN_THRESHOLD = 0.01       # minimum recency_jump_score to be considered


# ============================================
# 11. TOP-K WEAK SIGNALS (RECENCY+MAGNITUDE)
# ============================================

def rank_weak_signals(df_plot, score_col, label):
    df_rank = df_plot.copy()

    # Ensure valid numeric values
    df_rank = df_rank.dropna(subset=[score_col, "Count"]).copy()
    df_rank = df_rank.rename(columns={score_col: "score"})

    lower = WEAK_MIN_COUNT
    upper = WEAK_MIN_UPPER_COUNT

    # -------------------------------------------------
    # 1) Hard filter: Count bounds + score threshold
    # -------------------------------------------------
    candidates = df_rank[
        (df_rank["Count"] >= lower) &
        (df_rank["Count"] <= upper) &
        (df_rank["score"] >= SCORE_MIN_THRESHOLD)
    ].copy()

    print(
        f"[{label}] Candidates after filters: "
        f"Count in [{lower}, {upper}], {score_col} >= {SCORE_MIN_THRESHOLD} -> {len(candidates)} topics"
    )

    if candidates.empty:
        print(f"[{label}] No weak-signal candidates after Count + score filters.")
        return pd.DataFrame()

    # -------------------------------------------------
    # 2) Compute weak_signal_score within candidates
    # -------------------------------------------------
    c_min, c_max = candidates["Count"].min(), candidates["Count"].max()
    if c_max > c_min:
        count_norm = 1.0 - (candidates["Count"] - c_min) / (c_max - c_min)
    else:
        count_norm = pd.Series(1.0, index=candidates.index)

    s = candidates["score"]
    s_min, s_max = s.min(), s.max()
    if s_max > s_min:
        score_norm = (s - s_min) / (s_max - s_min)
    else:
        score_norm = pd.Series(1.0, index=candidates.index)

    BETA = 0.5  # weight between score and (inverted) Count
    candidates["weak_signal_score"] = (
        BETA * score_norm + (1.0 - BETA) * count_norm
    )

    # -------------------------------------------------
    # 3) Round-robin selection by Count (even filling)
    # -------------------------------------------------
    def round_robin_by_count(pool, k):
        if pool.empty or k <= 0:
            return []

        # For each Count, keep indices sorted by weak_signal_score desc
        groups = {}
        for c, sub in pool.groupby("Count"):
            sub_sorted = sub.sort_values("weak_signal_score", ascending=False)
            groups[c] = list(sub_sorted.index)

        selected = []
        pointers = {c: 0 for c in groups}

        # Classic round-robin: cycle over Counts, pick next best from each
        while len(selected) < k:
            any_picked = False
            for c in sorted(groups):  # sorted by Count for determinism
                idx_list = groups[c]
                pos = pointers[c]

                if pos < len(idx_list):
                    selected.append(idx_list[pos])
                    pointers[c] = pos + 1
                    any_picked = True

                    if len(selected) >= k:
                        break

            if not any_picked:
                # No group has remaining candidates
                break

        return selected

    selected_idx = round_robin_by_count(candidates, TOP_K_WEAK)

    if not selected_idx:
        print(f"[{label}] Round-robin could not select any topics.")
        return pd.DataFrame()

    ws_top = candidates.loc[selected_idx].copy()
    ws_top = ws_top.sort_values("weak_signal_score", ascending=False)

    cols = ["Name", "Count", "score", "weak_signal_score", "Representation"]
    ws_top = ws_top[cols].reset_index(drop=True)

    print(
        f"[{label}] Showing top {len(ws_top)} weak signals "
        f"(target TOP_K_WEAK={TOP_K_WEAK})."
    )
    return ws_top


# Only compute the Recency+Jump weak signals
ws_jump_top = rank_weak_signals(
    df_plot_jump,
    "recency_jump_score",
    label="Recency+Jump"
)

if not ws_jump_top.empty:
    pd.set_option("display.max_colwidth", None)
    display(ws_jump_top)
else:
    print("No weak signals to display for Recency+Jump scoring.")


In [48]:
import plotly.graph_objects as go
import re

# --- 1) ryr mask (red) ---
mask_ryr = df_plot_jump["Representation"].apply(
    lambda lst: any(
        re.fullmatch(r"(ryr)", str(x).lower())
        for x in lst
    )
)

# --- 2) Weak-signal mask from round-robin ranking (green) ---
# ws_jump_top was returned by rank_weak_signals(...)
weak_names = set(ws_jump_top["Name"])
mask_weak = df_plot_jump["Name"].isin(weak_names)

# Weak signals excluding ryr (so ryr stays only in red layer)
mask_weak_non_ryr = mask_weak & ~mask_ryr

# Base (neither ryr nor weak-signal)
mask_base = ~mask_ryr & ~mask_weak

df_base = df_plot_jump[mask_base]
df_weak = df_plot_jump[mask_weak_non_ryr]
df_ryr = df_plot_jump[mask_ryr]

fig = go.Figure()

# --- Base layer: all other topics ---
fig.add_trace(
    go.Scatter(
        x=df_base["Count"],
        y=df_base["recency_jump_score"],
        mode="markers",
        marker=dict(size=8, opacity=0.4),
        text=df_base["Name"],
        hovertext=df_base["Representation"].astype(str),
        name="Other topics",
    )
)

# --- Weak signals (round-robin) in green ---
fig.add_trace(
    go.Scatter(
        x=df_weak["Count"],
        y=df_weak["recency_jump_score"],
        mode="markers",
        marker=dict(
            size=10,
            color="green",
            line=dict(width=1, color="black"),
            opacity=0.9,
        ),
        text=df_weak["Name"],
        hovertext=df_weak["Representation"].astype(str),
        name="Weak signals",
    )
)

# --- ryr-related topic(s) in red ---
fig.add_trace(
    go.Scatter(
        x=df_ryr["Count"],
        y=df_ryr["recency_jump_score"],
        mode="markers",
        marker=dict(
            size=14,
            color="red",
            line=dict(width=2, color="black"),
        ),
        text=df_ryr["Name"],
        hovertext=df_ryr["Representation"].astype(str),
        name="RYR Weak Signal",
    )
)

# Axis + layout
fig.update_xaxes(title_text="Count")
fig.update_yaxes(title_text="recency_magnitude_score")  # label text only

fig.update_layout(
    title="Weak-signals: recency + magnitude of topics"
)

fig.show()
